# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution — Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library, following the Croissant schema. 

### Dataset Source
FAIR^2 dataset: Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution ([Croissant schema JSON-LD](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json))

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print("Dataset Title:", metadata.name)
print("Description:", metadata.description)
print("Published:", getattr(metadata, 'datePublished', 'N/A'))
print("Authors:")
for author in getattr(metadata, 'author', []):
    name = getattr(author, 'name', author)
    print(f"- {name}")

## 2. Data Overview
List available record sets (tables) in the dataset and their fields (columns), referencing by their `@id`.

In [ ]:
# List all RecordSets in this Croissant package and show their @id and description
record_sets = list(dataset.record_sets)
if not record_sets:
    raise RuntimeError('No record sets found in this dataset package.')

for rs in record_sets:
    print(f'RecordSet @id: {rs["@id"]}')
    print(f'  Name: {getattr(rs, "name", "<unnamed>")})')
    print(f'  Description: {getattr(rs, "description", "<no description>")})')
    print(f'  Fields:')
    for field in getattr(rs, 'fields', []):
        print(f"    - @id: {field['@id']}, Name: {getattr(field, 'name', '')}, DataType: {getattr(field, 'data_type', '')}")
    print()

## 3. Data Extraction
Load the main clinical records table into a DataFrame by referencing the RecordSet and Field `@id`.

In [ ]:
# Prepare to extract record sets into DataFrames
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for rs in record_sets:
    rs_id = rs['@id']
    # Load all records as a dict
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from RecordSet '@id': {rs_id}")
        print("Columns (@id):", list(dataframes[rs_id].columns))
        print()

# As an example, select the first RecordSet for further analysis
main_rs_id = record_set_ids[0]
print(f"Using RecordSet @id: {main_rs_id} for further demonstration.")
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field and a categorical field, filter rows, normalize values, and group by a category. All references are by `@id`.

In [ ]:
# Inspect columns to select suitable fields
df = dataframes[main_rs_id]
columns = list(df.columns)
print('Columns available in the record set:')
print(columns)

# For illustration, suppose the dataset contains the following numeric and categorical fields:
# Numeric field: '@id' for 'age' column
# Categorical field: '@id' for 'sex' column
# (Replace these with actual @id strings from your data)
possible_numeric_fields = [col for col in columns if 'age' in col.lower() or 'interval' in col.lower() or 'metastasis' in col.lower() or 'number' in col.lower()]
if not possible_numeric_fields:
    # fallback to first column as numeric example
    numeric_field = columns[0]
else:
    numeric_field = possible_numeric_fields[0]
print(f"Using numeric field: {numeric_field}")

possible_categorical_fields = [col for col in columns if 'sex' in col.lower() or 'location' in col.lower() or 'msi' in col.lower() or 'status' in col.lower() or 'type' in col.lower()] 
if not possible_categorical_fields:
    group_field = columns[1] if len(columns) > 1 else columns[0]
else:
    group_field = possible_categorical_fields[0]
print(f"Using grouping field: {group_field}")

# Convert to numeric where possible for the selected field
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
# Filter out rows where the value is missing
filtered_df = df[df[numeric_field].notnull()]

threshold = filtered_df[numeric_field].mean()
filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}):")
print(filtered_df[[numeric_field, group_field]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()

print(f"\nNormalized values for {numeric_field}:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouped statistics by categorical field (@id)
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].agg(['mean', 'count', 'std'])
    print(f"\nGrouped statistics by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship to the categorical/grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot of the numeric field by group/categorical field
if group_field in df.columns:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f'{numeric_field} by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion

- We loaded the FAIR^2 dataset using the Croissant schema and inspected its structure via RecordSets and Field IDs.
- A main RecordSet was loaded into a DataFrame, with columns referenced by `@id` per best practice.
- Example filtering and normalization steps were performed on a selected numeric field, and statistics grouped by a selected categorical field.
- Data distributions and group-level differences were visualized.

These steps serve as a foundation for deeper statistical or machine learning analysis on FAIR^2 and any Croissant-compatible dataset with `mlcroissant`.